In [2]:
import duckdb

# 1. Connect
con = duckdb.connect()

# 2. Load raw CSV 
con.execute("""
    CREATE TABLE movies_raw AS
    SELECT * FROM read_csv_auto(
        'IMDB TMDB Movie Metadata Big Dataset (1M).csv',
        ignore_errors=True
    )
""")

print("Raw shape:")
con.execute("SELECT COUNT(*), COUNT(DISTINCT title) FROM movies_raw").df()

Raw shape:


,count_star(),count(DISTINCT title)
0,1072255,921371


In [3]:
con.execute("""
    CREATE TABLE movies_clean AS
    SELECT
        title,
        CAST(release_year AS INTEGER)        AS release_year,
        CAST(runtime AS DOUBLE)              AS runtime,
        CAST(budget AS DOUBLE)               AS budget,
        CAST(revenue AS DOUBLE)              AS revenue,
        CAST("IMDB_Rating" AS DOUBLE)        AS imdb_rating,
        CAST(vote_count AS INTEGER)          AS vote_count,
        original_language,
        overview,
        tagline,
        keywords,
        genres_list,
        "Cast_list"                          AS cast_list,
        "Director"                           AS director,
        "Star1"                              AS actor,
        "Writer"                             AS writer,
        production_companies,
        production_countries,
        release_date,
        ROUND(((revenue - budget) / budget) * 100, 2) AS roi

    FROM movies_raw

    WHERE title           IS NOT NULL
    AND   "Director"      IS NOT NULL
    AND   "Star1"         IS NOT NULL
    AND   release_year    IS NOT NULL
    AND   revenue         > 0
    AND   budget          > 0
    AND   overview        IS NOT NULL AND overview != ''
    AND   tagline         IS NOT NULL AND tagline  != ''
""")

print("Clean shape:")
con.execute("SELECT COUNT(*) FROM movies_clean").df()

Clean shape:


,count_star()
0,3874


In [4]:
# ── 4. Verify no nulls ────────────────────────────────────────────────────────
con.execute("""
    SELECT
        COUNT(*) - COUNT(title)              AS title_nulls,
        COUNT(*) - COUNT(director)           AS director_nulls,
        COUNT(*) - COUNT(actor)            AS actor_nulls,
        COUNT(*) - COUNT(overview)           AS overview_nulls,
        COUNT(*) - COUNT(tagline)            AS tagline_nulls,
        COUNT(*) - COUNT(revenue)            AS revenue_nulls
    FROM movies_clean
""").df()

,title_nulls,director_nulls,actor_nulls,overview_nulls,tagline_nulls,revenue_nulls
0,0,0,0,0,0,0


In [5]:
# ── 5. Preview ────────────────────────────────────────────────────────────────
con.execute("SELECT * FROM movies_clean LIMIT 5").df()

,title,release_year,runtime,budget,revenue,imdb_rating,vote_count,original_language,overview,tagline,keywords,genres_list,cast_list,director,actor,writer,production_companies,production_countries,release_date,roi
0,Inception,2010,148.0,160000000.0,8.255328e+08,8.8,34495,en,"Cobb, a skilled thief who commits corporate es...",Your mind is the scene of the crime.,"['rescue', 'mission', 'dream', 'airplane', 'pa...","['Action', 'Science Fiction', 'Adventure']","['Tim Kelleher', 'Silvie Laguna', 'Natasha Bea...",Christopher Nolan,Leonardo DiCaprio,Christopher Nolan,"Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America",2010-07-15,415.96
1,Interstellar,2014,169.0,165000000.0,7.017292e+08,8.6,32571,en,The adventures of a group of explorers who mak...,Mankind was born on Earth. It was never meant ...,"['rescue', 'future', 'spacecraft', 'race again...","['Adventure', 'Drama', 'Science Fiction']","['Jeff Hephner', 'William Devane', 'Elyes Gabe...",Christopher Nolan,Matthew McConaughey,Jonathan Nolan,"Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",2014-11-05,325.29
2,The Dark Knight,2008,152.0,185000000.0,1.004558e+09,9.0,30619,en,Batman raises the stakes in his war on crime. ...,Welcome to a world without rules.,"['joker', 'sadism', 'chaos', 'secret identity'...","['Drama', 'Action', 'Crime', 'Thriller']","['Tommy Lister Jr.', 'Edison Chen', 'Beatrice ...",Christopher Nolan,Christian Bale,Jonathan Nolan,"DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America",2008-07-16,443.00
3,Avatar,2009,162.0,237000000.0,2.923706e+09,7.8,29815,en,"In the 22nd century, a paraplegic Marine is di...",Enter the world of Pandora.,"['future', 'society', 'culture clash', 'space ...","['Action', 'Adventure', 'Fantasy', 'Science Fi...","['Carvon Futrell', 'Joel David Moore', 'Jon Cu...",James Cameron,Sam Worthington,James Cameron,"Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom",2009-12-15,1133.63
4,The Avengers,2012,143.0,220000000.0,1.518816e+09,8.0,29166,en,When an unexpected enemy emerges and threatens...,Some assembly required.,"['new york city', 'superhero', 'shield', 'base...","['Science Fiction', 'Action', 'Adventure']","['Haneyuri', 'Nako Mizusawa', 'Marin', 'Rikako...",Joss Whedon,Robert Downey Jr.,Sydney Newman,Marvel Studios,United States of America,2012-04-25,590.37


In [6]:
# Dropping one column -cast_list-
con.execute("""
    ALTER TABLE movies_clean 
    DROP COLUMN cast_list
""")

print("Dropped!")
con.execute("SELECT * FROM movies_clean LIMIT 2").df()

Dropped!


,title,release_year,runtime,budget,revenue,imdb_rating,vote_count,original_language,overview,tagline,keywords,genres_list,director,actor,writer,production_companies,production_countries,release_date,roi
0,Inception,2010,148.0,160000000.0,825532764.0,8.8,34495,en,"Cobb, a skilled thief who commits corporate es...",Your mind is the scene of the crime.,"['rescue', 'mission', 'dream', 'airplane', 'pa...","['Action', 'Science Fiction', 'Adventure']",Christopher Nolan,Leonardo DiCaprio,Christopher Nolan,"Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America",2010-07-15,415.96
1,Interstellar,2014,169.0,165000000.0,701729206.0,8.6,32571,en,The adventures of a group of explorers who mak...,Mankind was born on Earth. It was never meant ...,"['rescue', 'future', 'spacecraft', 'race again...","['Adventure', 'Drama', 'Science Fiction']",Christopher Nolan,Matthew McConaughey,Jonathan Nolan,"Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",2014-11-05,325.29


In [7]:
con.execute("""
    ALTER TABLE movies_clean 
    DROP COLUMN keywords
""")

print("Dropped!")
con.execute("SELECT * FROM movies_clean LIMIT 2").df()

Dropped!


,title,release_year,runtime,budget,revenue,imdb_rating,vote_count,original_language,overview,tagline,genres_list,director,actor,writer,production_companies,production_countries,release_date,roi
0,Inception,2010,148.0,160000000.0,825532764.0,8.8,34495,en,"Cobb, a skilled thief who commits corporate es...",Your mind is the scene of the crime.,"['Action', 'Science Fiction', 'Adventure']",Christopher Nolan,Leonardo DiCaprio,Christopher Nolan,"Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America",2010-07-15,415.96
1,Interstellar,2014,169.0,165000000.0,701729206.0,8.6,32571,en,The adventures of a group of explorers who mak...,Mankind was born on Earth. It was never meant ...,"['Adventure', 'Drama', 'Science Fiction']",Christopher Nolan,Matthew McConaughey,Jonathan Nolan,"Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",2014-11-05,325.29


In [8]:
'''Deleting all entries which has Null values'''


con.execute("""
    DELETE FROM movies_clean
    WHERE title                = '' OR title                IS NULL
    OR    overview             = '' OR overview             IS NULL
    OR    tagline              = '' OR tagline              IS NULL
    OR    director             = '' OR director             IS NULL
    OR    actor                = '' OR actor                IS NULL
    OR    writer               = '' OR writer               IS NULL
    OR    genres_list          = '' OR genres_list          IS NULL
    OR    original_language    = '' OR original_language    IS NULL
    OR    production_companies = '' OR production_companies IS NULL
    OR    production_countries = '' OR production_countries IS NULL
    OR    release_date         IS NULL
    OR    roi                  IS NULL
    OR    budget               IS NULL
    OR    revenue              IS NULL
    OR    imdb_rating          IS NULL
    OR    runtime              IS NULL
    OR    vote_count           IS NULL
""")

print("Deleted!")
con.execute("SELECT COUNT(*) FROM movies_clean").df()

Deleted!


,count_star()
0,3809


In [9]:
"Verifying Nulls"

con.execute("""
    SELECT
        COUNT(*) - COUNT(title)                AS title_nulls,
        COUNT(*) - COUNT(release_year)         AS release_year_nulls,
        COUNT(*) - COUNT(runtime)              AS runtime_nulls,
        COUNT(*) - COUNT(budget)               AS budget_nulls,
        COUNT(*) - COUNT(revenue)              AS revenue_nulls,
        COUNT(*) - COUNT(imdb_rating)          AS imdb_rating_nulls,
        COUNT(*) - COUNT(vote_count)           AS vote_count_nulls,
        COUNT(*) - COUNT(overview)             AS overview_nulls,
        COUNT(*) - COUNT(tagline)              AS tagline_nulls,
        COUNT(*) - COUNT(director)             AS director_nulls,
        COUNT(*) - COUNT(actor)                AS actor_nulls,
        COUNT(*) - COUNT(writer)               AS writer_nulls,
        COUNT(*) - COUNT(roi)                  AS roi_nulls
    FROM movies_clean
""").df()

,title_nulls,release_year_nulls,runtime_nulls,budget_nulls,revenue_nulls,imdb_rating_nulls,vote_count_nulls,overview_nulls,tagline_nulls,director_nulls,actor_nulls,writer_nulls,roi_nulls
0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [10]:
'''using TRIM() — it strips whitespace before checking for empty strings'''

# ── Drop the empty table ──────────────────────────────────────────────────────
con.execute("DROP TABLE IF EXISTS movies_clean")

# ── Recreate with proper cleaning ─────────────────────────────────────────────
con.execute("""
    CREATE TABLE movies_clean AS
    SELECT
        title,
        CAST(release_year AS INTEGER)                  AS release_year,
        CAST(runtime AS DOUBLE)                        AS runtime,
        CAST(budget AS DOUBLE)                         AS budget,
        CAST(revenue AS DOUBLE)                        AS revenue,
        CAST("IMDB_Rating" AS DOUBLE)                  AS imdb_rating,
        CAST(vote_count AS INTEGER)                    AS vote_count,
        original_language,
        overview,
        tagline,
        genres_list,
        "Director"                                     AS director,
        "Star1"                                        AS actor,
        "Writer"                                       AS writer,
        production_companies,
        production_countries,
        release_date,
        ROUND(((revenue - budget) / budget) * 100, 2) AS roi

    FROM movies_raw

    WHERE revenue        > 0
    AND   budget         > 0
    AND   "Director"     IS NOT NULL AND TRIM("Director") != ''
    AND   "Star1"        IS NOT NULL AND TRIM("Star1")    != ''
    AND   title          IS NOT NULL AND TRIM(title)      != ''
    AND   overview       IS NOT NULL AND TRIM(overview)   != ''
    AND   tagline        IS NOT NULL AND TRIM(tagline)    != ''
    AND   "Writer"       IS NOT NULL AND TRIM("Writer")   != ''
    AND   release_year   IS NOT NULL
""")

print("Clean shape:")
con.execute("SELECT COUNT(*) FROM movies_clean").df()

Clean shape:


,count_star()
0,3832


In [11]:
'''verifying null one last time'''

con.execute("""
    SELECT
        COUNT(*) - COUNT(title)                AS title_nulls,
        COUNT(*) - COUNT(overview)             AS overview_nulls,
        COUNT(*) - COUNT(tagline)              AS tagline_nulls,
        COUNT(*) - COUNT(director)             AS director_nulls,
        COUNT(*) - COUNT(actor)                AS actor_nulls,
        COUNT(*) - COUNT(writer)               AS writer_nulls,
        COUNT(*) - COUNT(budget)               AS budget_nulls,
        COUNT(*) - COUNT(revenue)              AS revenue_nulls,
        COUNT(*) - COUNT(imdb_rating)          AS imdb_rating_nulls,
        COUNT(*) - COUNT(roi)                  AS roi_nulls
    FROM movies_clean
""").df()

,title_nulls,overview_nulls,tagline_nulls,director_nulls,actor_nulls,writer_nulls,budget_nulls,revenue_nulls,imdb_rating_nulls,roi_nulls
0,0,0,0,0,0,0,0,0,0,0


In [12]:
'''Checking Duplicates'''

con.execute("""
    SELECT title, release_year, COUNT(*) as count
    FROM movies_clean
    GROUP BY title, release_year
    HAVING COUNT(*) > 1
    ORDER BY count DESC
    LIMIT 10
""").df()

,title,release_year,count


In [13]:
'''Check for Unrealistics Values'''

con.execute("""
    SELECT 
        MIN(budget)       AS min_budget,
        MAX(budget)       AS max_budget,
        MIN(revenue)      AS min_revenue,
        MAX(revenue)      AS max_revenue,
        MIN(imdb_rating)  AS min_rating,
        MAX(imdb_rating)  AS max_rating,
        MIN(runtime)      AS min_runtime,
        MAX(runtime)      AS max_runtime,
        MIN(release_year) AS min_year,
        MAX(release_year) AS max_year
    FROM movies_clean
""").df()

,min_budget,max_budget,min_revenue,max_revenue,min_rating,max_rating,min_runtime,max_runtime,min_year,max_year
0,1.0,460000000.0,1.0,2.923706e+09,2.1,9.3,0.0,248.0,1915,2024


In [14]:
'''Checking language Distribution'''

con.execute("""
    SELECT original_language, COUNT(*) as count
    FROM movies_clean
    GROUP BY original_language
    ORDER BY count DESC
    LIMIT 10
""").df()

,original_language,count
0,en,3565
1,fr,49
2,ja,34
3,ko,34
4,es,31
5,hi,22
6,it,15
7,ru,14
8,de,10
9,cn,6


In [15]:
'''deleting movies which are realeased before 1990'''

con.execute("""
    DELETE FROM movies_clean
    WHERE budget    <= 1000
    OR    revenue   <= 1000
    OR    runtime   <= 0
    OR    release_year < 1990
""")

print("Cleaned!")
con.execute("SELECT COUNT(*) FROM movies_clean").df()

Cleaned!


,count_star()
0,3146


In [30]:
# ── 6. Save as CSV ────────────────────────────────────────────────────────────
con.execute("""
    COPY movies_clean TO 'movies_cleaned.csv' (HEADER, DELIMITER ',')
""")
print("Saved as movies_cleaned.csv")

Saved as movies_cleaned.csv


In [17]:
con.execute("DESCRIBE movies_clean").df()

,column_name,column_type,null,key,default,extra
0,title,VARCHAR,YES,None,None,None
1,release_year,INTEGER,YES,None,None,None
2,runtime,DOUBLE,YES,None,None,None
3,budget,DOUBLE,YES,None,None,None
4,revenue,DOUBLE,YES,None,None,None
5,imdb_rating,DOUBLE,YES,None,None,None
6,vote_count,INTEGER,YES,None,None,None
7,original_language,VARCHAR,YES,None,None,None
8,overview,VARCHAR,YES,None,None,None
9,tagline,VARCHAR,YES,None,None,None


In [18]:
'''Analytical Questions that answer my bussiness question'''

# ── 1. Top director-actor collaborations by revenue ───────────────────────────
con.execute("""
    SELECT 
        director,
        actor,
        COUNT(*)                        AS movies_together,
        ROUND(AVG(revenue), 2)          AS avg_revenue,
        ROUND(SUM(revenue), 2)          AS total_revenue,
        ROUND(AVG(imdb_rating), 2)      AS avg_rating,
        ROUND(AVG(roi), 2)              AS avg_roi
    FROM movies_clean
    GROUP BY director, actor
    HAVING COUNT(*) >= 2
    ORDER BY total_revenue DESC
    LIMIT 10
""").df()

,director,actor,movies_together,avg_revenue,total_revenue,avg_rating,avg_roi
0,"Anthony Russo, Joe Russo",Joe Russo,3,1.855727e+09,5.567182e+09,8.17,530.37
1,Christopher Nolan,Christian Bale,4,9.188489e+08,3.675396e+09,8.50,358.17
2,David Yates,Daniel Radcliffe,3,1.076592e+09,3.229776e+09,7.80,509.50
3,Peter Jackson,Elijah Wood,3,9.721816e+08,2.916545e+09,8.80,999.93
4,Sam Mendes,Daniel Craig,2,9.052584e+08,1.810517e+09,7.70,334.08
5,Robert Zemeckis,Tom Hanks,2,8.269316e+08,1.653863e+09,8.80,906.40
6,James Gunn,Chris Pratt,2,8.182663e+08,1.636533e+09,7.80,343.23
7,George Lucas,Tom Hanks,2,6.773084e+08,1.354617e+09,8.05,592.08
8,Alfonso Cuarón,Daniel Radcliffe,2,6.195656e+08,1.239131e+09,7.90,353.55
9,Park Chan-Wook,Choi Min-sik,2,4.863544e+08,9.727088e+08,8.40,580.89


In [19]:
''' Query 1 — Top Director-Actor Collaborations by Revenue
Russo Brothers + Joe Russo generated the highest total revenue of $5.57B across just 3 films with an impressive 
530% ROI and 8.17 rating. Christopher Nolan + Christian Bale collaborated the most (4 films) with $3.67B total 
and highest avg rating of 8.50. Peter Jackson + Elijah Wood stand out with the best balance — 8.80 rating AND 999% ROI. 
This confirms that long-term director-actor partnerships are a reliable formula for commercial success.   '''

' Query 1 — Top Director-Actor Collaborations by Revenue\nRusso Brothers + Joe Russo generated the highest total revenue of $5.57B across just 3 films with an impressive \n530% ROI and 8.17 rating. Christopher Nolan + Christian Bale collaborated the most (4 films) with $3.67B total \nand highest avg rating of 8.50. Peter Jackson + Elijah Wood stand out with the best balance — 8.80 rating AND 999% ROI. \nThis confirms that long-term director-actor partnerships are a reliable formula for commercial success.   '

In [20]:
# ── 2. Most commercially successful directors ─────────────────────────────────
con.execute("""
    SELECT
        director,
        COUNT(*)                        AS total_movies,
        ROUND(AVG(revenue), 2)          AS avg_revenue,
        ROUND(SUM(revenue), 2)          AS total_revenue,
        ROUND(AVG(imdb_rating), 2)      AS avg_rating
    FROM movies_clean
    GROUP BY director
    HAVING COUNT(*) >= 3
    ORDER BY total_revenue DESC
    LIMIT 10
""").df()

,director,total_movies,avg_revenue,total_revenue,avg_rating
0,Christopher Nolan,15,5.425486e+08,8.138229e+09,8.02
1,Steven Spielberg,27,2.718208e+08,7.339160e+09,7.37
2,James Cameron,7,9.212180e+08,6.448526e+09,7.74
3,Robert Zemeckis,15,3.940515e+08,5.910772e+09,7.40
4,"Anthony Russo, Joe Russo",3,1.855727e+09,5.567182e+09,8.17
5,Ridley Scott,21,2.503433e+08,5.257209e+09,6.99
6,Peter Jackson,11,4.687294e+08,5.156023e+09,7.73
7,David Yates,6,7.536318e+08,4.521791e+09,7.75
8,M. Night Shyamalan,12,3.699272e+08,4.439127e+09,6.75
9,Tim Burton,20,2.165210e+08,4.330420e+09,7.39


In [21]:
'''Query 2 — Most Commercially Successful Directors

Christopher Nolan leads with $8.14B total revenue and highest avg rating of 8.02 across 15 films — the most consistent 
director overall.

Steven Spielberg is the most prolific with 27 films and $7.34B total — but his lower avg revenue of $271M per film 
shows he relies on consistency rather than mega-blockbusters.

James Cameron has only 7 films but dominates with $921M avg revenue per film — the highest per-film return in the 
entire list.

Russo Brothers are the most efficient — only 3 films but already $5.57B total with the highest avg rating of 8.17.

David Yates owes his $753M avg revenue entirely to the Harry Potter franchise — proving that franchise loyalty 
is one of the most commercially reliable formulas.'''

'Query 2 — Most Commercially Successful Directors\n\nChristopher Nolan leads with $8.14B total revenue and highest avg rating of 8.02 across 15 films — the most consistent \ndirector overall.\n\nSteven Spielberg is the most prolific with 27 films and $7.34B total — but his lower avg revenue of $271M per film \nshows he relies on consistency rather than mega-blockbusters.\n\nJames Cameron has only 7 films but dominates with $921M avg revenue per film — the highest per-film return in the \nentire list.\n\nRusso Brothers are the most efficient — only 3 films but already $5.57B total with the highest avg rating of 8.17.\n\nDavid Yates owes his $753M avg revenue entirely to the Harry Potter franchise — proving that franchise loyalty \nis one of the most commercially reliable formulas.'

In [22]:
# ── 3. Most successful genres by revenue ──────────────────────────────────────
con.execute("""
    SELECT
        genres_list,
        COUNT(*)                        AS total_movies,
        ROUND(AVG(revenue), 2)          AS avg_revenue,
        ROUND(AVG(imdb_rating), 2)      AS avg_rating,
        ROUND(AVG(roi), 2)              AS avg_roi
    FROM movies_clean
    GROUP BY genres_list
    HAVING COUNT(*) >= 5
    ORDER BY avg_revenue DESC
    LIMIT 10
""").df()

,genres_list,total_movies,avg_revenue,avg_rating,avg_roi
0,"['Adventure', 'Action', 'Science Fiction']",17,8.361415e+08,7.25,423.44
1,"['Adventure', 'Fantasy']",8,7.391960e+08,7.46,393.12
2,"['Adventure', 'Fantasy', 'Action']",13,6.210023e+08,7.19,379.61
3,"['Science Fiction', 'Adventure', 'Action']",8,6.076422e+08,7.10,133.82
4,"['Action', 'Science Fiction', 'Adventure']",7,5.947317e+08,7.89,260.90
5,"['Action', 'Adventure', 'Fantasy']",17,5.382124e+08,7.46,208.98
6,"['Action', 'Adventure', 'Science Fiction']",36,5.190416e+08,7.26,220.00
7,"['Fantasy', 'Adventure', 'Family']",5,4.845081e+08,7.46,155.21
8,"['Animation', 'Family']",10,4.587675e+08,7.05,322.97
9,"['Animation', 'Comedy', 'Family', 'Adventure']",5,4.538551e+08,6.30,416.55


In [23]:
''' Query 3 — Most Successful Genres
Adventure + Action + Sci-Fi dominates with avg revenue of $836M and 423% ROI. Every top genre combination 
includes either Adventure or Action — pure drama or comedy doesn't appear at all. This tells studios exactly 
which genre combinations to invest in.'''

" Query 3 — Most Successful Genres\nAdventure + Action + Sci-Fi dominates with avg revenue of $836M and 423% ROI. Every top genre combination \nincludes either Adventure or Action — pure drama or comedy doesn't appear at all. This tells studios exactly \nwhich genre combinations to invest in."

In [24]:
# ── 4. Revenue trend over years ───────────────────────────────────────────────
con.execute("""
    SELECT
        release_year,
        COUNT(*)                        AS total_movies,
        ROUND(AVG(revenue), 2)          AS avg_revenue,
        ROUND(AVG(imdb_rating), 2)      AS avg_rating
    FROM movies_clean
    GROUP BY release_year
    ORDER BY release_year
""").df()

,release_year,total_movies,avg_revenue,avg_rating
0,1990,40,1.146586e+08,7.01
1,1991,46,9.341681e+07,6.62
2,1992,47,1.014510e+08,6.78
3,1993,55,9.551695e+07,6.87
4,1994,47,1.288758e+08,6.85
5,1995,61,9.278752e+07,6.78
6,1996,54,1.031179e+08,6.61
7,1997,76,1.446782e+08,7.09
8,1998,85,1.075355e+08,6.76
9,1999,71,1.300907e+08,6.78


In [25]:
'''Query 4 — Revenue Trend Over Years
Revenue was relatively flat (~$100-140M avg) through the 1990s-2000s, with production volume steadily increasing 
from 40 movies in 1990 to 100+ by 2002. This sets up an interesting story about how franchise filmmaking changed 
the industry post-2000.'''

'Query 4 — Revenue Trend Over Years\nRevenue was relatively flat (~$100-140M avg) through the 1990s-2000s, with production volume steadily increasing \nfrom 40 movies in 1990 to 100+ by 2002. This sets up an interesting story about how franchise filmmaking changed \nthe industry post-2000.'

In [26]:
# ── 5. High ROI collaborations ────────────────────────────────────────────────
con.execute("""
    SELECT
        director,
        actor,
        COUNT(*)                        AS movies_together,
        ROUND(AVG(roi), 2)              AS avg_roi,
        ROUND(AVG(revenue), 2)          AS avg_revenue,
        ROUND(AVG(imdb_rating), 2)      AS avg_rating
    FROM movies_clean
    GROUP BY director, actor
    HAVING COUNT(*) >= 2
    ORDER BY avg_roi DESC
    LIMIT 10
""").df()

,director,actor,movies_together,avg_roi,avg_revenue,avg_rating
0,Michel Hazanavicius,Jean Dujardin,2,22325.61,8.100397e+07,7.50
1,Steven Soderbergh,Julia Roberts,2,6365.92,7.501654e+07,6.05
2,Peter Jackson,Elijah Wood,3,999.93,9.721816e+08,8.80
3,Robert Zemeckis,Tom Hanks,2,906.40,8.269316e+08,8.80
4,Tom McCarthy,Peter Dinklage,2,811.40,2.952260e+07,7.60
5,Steven Soderbergh,Ryan Reynolds,2,775.94,2.649693e+08,6.20
6,George Lucas,Tom Hanks,2,592.08,6.773084e+08,8.05
7,Park Chan-Wook,Choi Min-sik,2,580.89,4.863544e+08,8.40
8,"Anthony Russo, Joe Russo",Joe Russo,3,530.37,1.855727e+09,8.17
9,David Yates,Daniel Radcliffe,3,509.50,1.076592e+09,7.80


In [27]:
'''Query 5 — High ROI Collaborations
Michel Hazanavicius + Jean Dujardin have an extraordinary 22,325% ROI — these are low-budget art films
that massively outperformed expectations. Peter Jackson + Elijah Wood deliver both high ROI (999%) AND 
high revenue ($972M avg) — the rarest and most valuable combination.'''

'Query 5 — High ROI Collaborations\nMichel Hazanavicius + Jean Dujardin have an extraordinary 22,325% ROI — these are low-budget art films\nthat massively outperformed expectations. Peter Jackson + Elijah Wood deliver both high ROI (999%) AND \nhigh revenue ($972M avg) — the rarest and most valuable combination.'

In [28]:
'''

Overall Conclusion
Collaboration networks between actors and directors have a measurable and significant impact on the 
commercial success of films.

Key Findings:

1. Repeated collaborations outperform one-off pairings

Directors and actors who work together multiple times consistently generate higher revenue and ratings. 
Russo Brothers + Joe Russo ($5.57B), Nolan + Bale (4 films, $3.67B) and Jackson + Elijah Wood (3 films, $2.9B) all prove this.

2. Quality directors are more valuable than prolific ones

Christopher Nolan with 15 films generated MORE total revenue than Steven Spielberg's 27 films — proving that director 
quality and consistency matters more than volume.

3. Genre combination is a multiplier

Adventure + Action + Sci-Fi generates 423% ROI on average — studios that pair strong collaboration networks WITH the right genre 
combination maximize both revenue and returns.

4. ROI and revenue tell different stories

High revenue = blockbuster franchises (Marvel, Harry Potter, Lord of the Rings)
High ROI = low-budget films that massively outperformed (Hazanavicius + Dujardin at 22,325%)
The best collaborations score high on both — Peter Jackson + Elijah Wood (999% ROI + $972M avg revenue)

5. The industry shifted post-2000

Revenue grew steadily from 1990s (~$100M avg) as franchise filmmaking took over — the same director-actor networks appear 
repeatedly in top franchises.

->Business Recommendation

A studio looking to maximize commercial success should invest in proven director-actor pairs working in Adventure/Action/Sci-Fi 
genres — this combination consistently delivers both high revenue and strong ratings.'''

"\n\nOverall Conclusion\nCollaboration networks between actors and directors have a measurable and significant impact on the \ncommercial success of films.\n\nKey Findings:\n\n1. Repeated collaborations outperform one-off pairings\n\nDirectors and actors who work together multiple times consistently generate higher revenue and ratings. \nRusso Brothers + Joe Russo ($5.57B), Nolan + Bale (4 films, $3.67B) and Jackson + Elijah Wood (3 films, $2.9B) all prove this.\n\n2. Quality directors are more valuable than prolific ones\n\nChristopher Nolan with 15 films generated MORE total revenue than Steven Spielberg's 27 films — proving that director \nquality and consistency matters more than volume.\n\n3. Genre combination is a multiplier\n\nAdventure + Action + Sci-Fi generates 423% ROI on average — studios that pair strong collaboration networks WITH the right genre \ncombination maximize both revenue and returns.\n\n4. ROI and revenue tell different stories\n\nHigh revenue = blockbuster fra

---
## 🔧 S3 Extension – Full Schema CSV Exports
_Added by Rauf — Hands-On 2 requirements_

Exporting separate CSV files for each **node label** and **relationship type**,
so that `02_graph_load.ipynb` can load the full schema via `LOAD CSV + MERGE`.

All staging tables use `CREATE OR REPLACE TABLE` for idempotency.

In [ ]:
# ── IDEMPOTENCY FIX: Rebuild movies_clean with CREATE OR REPLACE TABLE ────────
# (replaces original CREATE TABLE so pipeline is safe to re-run)

con.execute("""
    CREATE OR REPLACE TABLE movies_clean AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY title, release_year) AS movie_id,
        title,
        CAST(release_year AS INTEGER)                  AS release_year,
        CAST(runtime AS DOUBLE)                        AS runtime,
        CAST(budget AS DOUBLE)                         AS budget,
        CAST(revenue AS DOUBLE)                        AS revenue,
        CAST("IMDB_Rating" AS DOUBLE)                  AS imdb_rating,
        CAST(vote_count AS INTEGER)                    AS vote_count,
        original_language,
        overview,
        tagline,
        genres_list,
        "Director"                                     AS director,
        "Star1"                                        AS actor,
        "Writer"                                       AS writer,
        production_companies,
        production_countries,
        release_date,
        ROUND(((revenue - budget) / budget) * 100, 2) AS roi
    FROM movies_raw
    WHERE revenue        > 1000
    AND   budget         > 1000
    AND   runtime        > 0
    AND   release_year   >= 1990
    AND   "Director"     IS NOT NULL AND TRIM("Director") != ''
    AND   "Star1"        IS NOT NULL AND TRIM("Star1")    != ''
    AND   title          IS NOT NULL AND TRIM(title)      != ''
    AND   overview       IS NOT NULL AND TRIM(overview)   != ''
    AND   tagline        IS NOT NULL AND TRIM(tagline)    != ''
    AND   "Writer"       IS NOT NULL AND TRIM("Writer")   != ''
""")

print("✅ movies_clean rebuilt with CREATE OR REPLACE TABLE (idempotent)")
con.execute("SELECT COUNT(*) AS total_rows FROM movies_clean").df()

In [ ]:
# ── NODE: Movie ───────────────────────────────────────────────────────────────
# Exports one row per unique movie with stable movie_id as key

con.execute("""
    COPY (
        SELECT DISTINCT
            movie_id,
            title,
            release_year,
            runtime,
            budget,
            revenue,
            imdb_rating,
            vote_count,
            original_language,
            overview,
            tagline,
            release_date,
            roi
        FROM movies_clean
    ) TO 'data/clean/Movie_nodes.csv' (HEADER, FORMAT CSV)
""")

print("✅ Movie_nodes.csv exported")
con.execute("SELECT COUNT(*) AS movie_count FROM movies_clean").df()

In [ ]:
# ── NODE: Director ────────────────────────────────────────────────────────────
# One row per unique director with stable director_id

con.execute("""
    CREATE OR REPLACE TABLE directors_staging AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY director) AS director_id,
        director AS name
    FROM (SELECT DISTINCT director FROM movies_clean)
""")

con.execute("""
    COPY directors_staging
    TO 'data/clean/Director_nodes.csv' (HEADER, FORMAT CSV)
""")

print("✅ Director_nodes.csv exported")
con.execute("SELECT COUNT(*) AS director_count FROM directors_staging").df()

In [ ]:
# ── NODE: Actor ───────────────────────────────────────────────────────────────
# One row per unique actor with stable actor_id

con.execute("""
    CREATE OR REPLACE TABLE actors_staging AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY actor) AS actor_id,
        actor AS name
    FROM (SELECT DISTINCT actor FROM movies_clean)
""")

con.execute("""
    COPY actors_staging
    TO 'data/clean/Actor_nodes.csv' (HEADER, FORMAT CSV)
""")

print("✅ Actor_nodes.csv exported")
con.execute("SELECT COUNT(*) AS actor_count FROM actors_staging").df()

In [ ]:
# ── NODE: Genre ───────────────────────────────────────────────────────────────
# One row per unique genre combination with stable genre_id

con.execute("""
    CREATE OR REPLACE TABLE genres_staging AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY genres_list) AS genre_id,
        genres_list AS name
    FROM (SELECT DISTINCT genres_list FROM movies_clean WHERE genres_list IS NOT NULL)
""")

con.execute("""
    COPY genres_staging
    TO 'data/clean/Genre_nodes.csv' (HEADER, FORMAT CSV)
""")

print("✅ Genre_nodes.csv exported")
con.execute("SELECT COUNT(*) AS genre_count FROM genres_staging").df()

In [ ]:
# ── RELATIONSHIP: DIRECTED (Director -> Movie) ────────────────────────────────
# src_id = director_id, dst_id = movie_id

con.execute("""
    COPY (
        SELECT
            d.director_id  AS src_id,
            m.movie_id     AS dst_id
        FROM movies_clean m
        JOIN directors_staging d ON d.name = m.director
    ) TO 'data/clean/DIRECTED.csv' (HEADER, FORMAT CSV)
""")

print("✅ DIRECTED.csv exported")
con.execute("SELECT COUNT(*) AS directed_count FROM movies_clean").df()

In [ ]:
# ── RELATIONSHIP: ACTED_IN (Actor -> Movie) ───────────────────────────────────
# src_id = actor_id, dst_id = movie_id

con.execute("""
    COPY (
        SELECT
            a.actor_id   AS src_id,
            m.movie_id   AS dst_id
        FROM movies_clean m
        JOIN actors_staging a ON a.name = m.actor
    ) TO 'data/clean/ACTED_IN.csv' (HEADER, FORMAT CSV)
""")

print("✅ ACTED_IN.csv exported")
con.execute("SELECT COUNT(*) AS acted_in_count FROM movies_clean").df()

In [ ]:
# ── RELATIONSHIP: BELONGS_TO (Movie -> Genre) ─────────────────────────────────
# src_id = movie_id, dst_id = genre_id

con.execute("""
    COPY (
        SELECT
            m.movie_id   AS src_id,
            g.genre_id   AS dst_id
        FROM movies_clean m
        JOIN genres_staging g ON g.name = m.genres_list
    ) TO 'data/clean/BELONGS_TO.csv' (HEADER, FORMAT CSV)
""")

print("✅ BELONGS_TO.csv exported")
con.execute("SELECT COUNT(*) AS belongs_to_count FROM movies_clean").df()

In [ ]:
# ── RELATIONSHIP: COLLABORATED_WITH (Director -> Actor) ───────────────────────
# src_id = director_id, dst_id = actor_id, with revenue + movie_count properties

con.execute("""
    CREATE OR REPLACE TABLE collaborated_staging AS
    SELECT
        d.director_id                   AS src_id,
        a.actor_id                      AS dst_id,
        COUNT(*)                        AS movie_count,
        ROUND(SUM(m.revenue), 2)        AS total_revenue
    FROM movies_clean m
    JOIN directors_staging d ON d.name = m.director
    JOIN actors_staging    a ON a.name = m.actor
    GROUP BY d.director_id, a.actor_id
""")

con.execute("""
    COPY collaborated_staging
    TO 'data/clean/COLLABORATED_WITH.csv' (HEADER, FORMAT CSV)
""")

print("✅ COLLABORATED_WITH.csv exported")
con.execute("SELECT COUNT(*) AS collab_count FROM collaborated_staging").df()

In [ ]:
# ── EXPORT SUMMARY ────────────────────────────────────────────────────────────
# Verify all files were created correctly

import os

files = [
    'data/clean/Movie_nodes.csv',
    'data/clean/Director_nodes.csv',
    'data/clean/Actor_nodes.csv',
    'data/clean/Genre_nodes.csv',
    'data/clean/DIRECTED.csv',
    'data/clean/ACTED_IN.csv',
    'data/clean/BELONGS_TO.csv',
    'data/clean/COLLABORATED_WITH.csv',
]

print("📁 Export Summary:")
print("-" * 45)
for f in files:
    if os.path.exists(f):
        size_kb = os.path.getsize(f) // 1024
        print(f"✅ {f.split('/')[-1]:35s} {size_kb} KB")
    else:
        print(f"❌ {f.split('/')[-1]:35s} NOT FOUND")
print("-" * 45)
print("Ready for 02_graph_load.ipynb")